# Camada Silver

Tratativas com o objetivo de padronizar os dados e deixa-los limpos e organizados. 

# 1. Tratativas tabela dados_saeb


Na tabela de dados do saeb os principais tratamentos realizados são:


1) Criar coluna ESTADO com o nome do estado, não um código, com o objetivo de facilitar o trabalho dos usuários finais. O de para de código para o nome do estado está no comentário da coluna na camada bronze

2) Criar a coluna area, mostrando se é capital ou interior a partir da coluna ID_AREA. Sendo 1 para Capital e 2 Interior

3) criar a coluna escola_publica, informando se é publica ou privada a partir da coluna IN_PUBLICA. Sendo 0 Privada e 1 Pública

4) criar a coluna localizacao, informando se é rural ou urbana a partir da coluna ID_LOCALIZACAO. sendo 1 Urbana e 2 Rural

5) Criação de colunas de FLAG, informando se a escola participou ou não do exame naquela faixa de ensino, sendo 1 para tendo participado e 0 para não

6) Criação de colunas de proficiência por ano escolar e disciplina, com o objetivo de traduzir as pontuações dos exames, que variam de 0 a 500 e possuem faixas de desempenho específicas para cada disciplina e ano escolar, em níveis de proficiência de interpretação mais clara, conforme a regra abaixo:

As médias de proficiência das escolas são classificadas de acordo com os
padrões de desempenho definidos para cada etapa de ensino e disciplina.

A classificação é realizada separadamente para:

- **5º ano do Ensino Fundamental**
- **9º ano do Ensino Fundamental**
- **3ª série do Ensino Médio**

E para as disciplinas:

- **Língua Portuguesa (LP)**
- **Matemática (MT)**

Quando a escola não apresenta resultado para determinada etapa/disciplina,
a classificação recebe o valor **"Sem resultado"**. Dessa forma, a ausência
de uma etapa de ensino não é interpretada como baixo desempenho.

---

###  Faixas de Avaliação de Matemática

| Padrão de Desempenho | 5º ano EF | 9º ano EF | 3ª série EM |
|---|---:|---:|---:|
| **Abaixo do Básico** | Até 175 pontos | Até 225 pontos | Até 275 pontos |
| **Básico** | 176 a 225 pontos | 226 a 300 pontos | 276 a 350 pontos |
| **Adequado** | 226 a 275 pontos | 301 a 350 pontos | 351 a 400 pontos |
| **Avançado** | 276 pontos ou mais | 351 pontos ou mais | 401 pontos ou mais |

---

###  Faixas de Avaliação de Língua Portuguesa

| Padrão de Desempenho | 5º ano EF | 9º ano EF | 3ª série EM |
|---|---:|---:|---:|
| **Abaixo do Básico** | Até 150 pontos | Até 225 pontos | Até 250 pontos |
| **Básico** | 151 a 200 pontos | 226 a 275 pontos | 251 a 300 pontos |
| **Adequado** | 201 a 250 pontos | 276 a 325 pontos | 301 a 350 pontos |
| **Avançado** | 251 pontos ou mais | 326 pontos ou mais | 351 pontos ou mais |

---


7) Apagar a coluna ID_REGIAO. Ela estará numa tabela dimensão de estado, que apresentará essas informações gerais por estado

8) Apagar as colunas anteriores codificadas que não serão mais utilizadas (ID_AREA, IN_PUBLICA, ID_LOCALIZACAO, ID_REGIAO )


## 1.0. Preparação

In [0]:
from pyspark.sql.functions import when, col

In [0]:
df_saeb = spark.table("`mvp-eng-dados-puc-rio`.bronze.resultados_saeb")

## 1.1 Coluna ESTADO

Criar coluna ESTADO com o nome do estado, não um código, com o objetivo de facilitar para o usuário final. O de para de código para o nome do estado está no comentário da coluna na camada bronze

In [0]:
# 2. Criar a coluna ESTADO
# Converte o código da UF (ID_UF) para o nome do estado

df_saeb = df_saeb.withColumn(
    "ESTADO",
    when(col("ID_UF") == 53, "Distrito Federal")
    .when(col("ID_UF") == 52, "Goiás")
    .when(col("ID_UF") == 51, "Mato Grosso")
    .when(col("ID_UF") == 50, "Mato Grosso do Sul")
    .when(col("ID_UF") == 27, "Alagoas")
    .when(col("ID_UF") == 29, "Bahia")
    .when(col("ID_UF") == 23, "Ceará")
    .when(col("ID_UF") == 21, "Maranhão")
    .when(col("ID_UF") == 25, "Paraíba")
    .when(col("ID_UF") == 26, "Pernambuco")
    .when(col("ID_UF") == 22, "Piauí")
    .when(col("ID_UF") == 24, "Rio Grande do Norte")
    .when(col("ID_UF") == 28, "Sergipe")
    .when(col("ID_UF") == 17, "Tocantins")
    .when(col("ID_UF") == 12, "Acre")
    .when(col("ID_UF") == 16, "Amapá")
    .when(col("ID_UF") == 13, "Amazonas")
    .when(col("ID_UF") == 15, "Pará")
    .when(col("ID_UF") == 11, "Rondônia")
    .when(col("ID_UF") == 14, "Roraima")
    .when(col("ID_UF") == 32, "Espírito Santo")
    .when(col("ID_UF") == 31, "Minas Gerais")
    .when(col("ID_UF") == 33, "Rio de Janeiro")
    .when(col("ID_UF") == 35, "São Paulo")
    .when(col("ID_UF") == 41, "Paraná")
    .when(col("ID_UF") == 43, "Rio Grande do Sul")
    .when(col("ID_UF") == 42, "Santa Catarina")
    .otherwise("Não identificado")
)


### 1.1.1. Qualidade dos Dados

In [0]:
df_saeb.filter(col("ESTADO") == "Não identificado").select("ID_UF").distinct().show()   


# criada com o intuito de verificar se há algum ID_UF que não estava na lista. Como resultado não foi identificado nenhum ID_UF faltante, a transformação foi feita corretamente


+-----+
|ID_UF|
+-----+
+-----+



## 1.2. Coluna area

Criar a coluna area, mostrando se é capital ou interior a partir da coluna ID_AREA. Sendo 1 para Capital e 2 Interior

In [0]:
# 3. Criar a coluna area
# ID_AREA = 1 → Capital
# ID_AREA = 2 → Interior

df_saeb = df_saeb.withColumn(
    "area",
    when(col("ID_AREA") == 1, "Capital")
    .when(col("ID_AREA") == 2, "Interior")
    .otherwise("Não identificado")
)


### 1.2.1. Qualidade dos Dados

In [0]:
df_saeb.select("area").distinct().show() 

# Tratamento para verificar se há algum valor faltante. Como só há valores Interior e Capital, a transformação foi correta


+--------+
|    area|
+--------+
|Interior|
| Capital|
+--------+



## 1.3. Coluna escola_publica

criar a coluna escola_publica, informando se é publica ou privada a partir da coluna IN_PUBLICA. Sendo 0 Privada e 1 Pública


In [0]:
# 4. Criar a coluna escola_publica
# IN_PUBLICA = 0 → Privada
# IN_PUBLICA = 1 → Pública

df_saeb = df_saeb.withColumn(
    "escola_publica",
    when(col("IN_PUBLICA") == 0, "Privada")
    .when(col("IN_PUBLICA") == 1, "Pública")
    .otherwise("Não identificado")
)

### 1.3.1. Qualidade dos Dados

In [0]:
df_saeb.select("escola_publica").distinct().show() 

# Tratamento para verificar se há algum valor faltante. Como só há valores Pública chama a atenção e será aprofundanda a verificação abaixo


+--------------+
|escola_publica|
+--------------+
|       Pública|
+--------------+



In [0]:
df_saeb.groupBy(
    "IN_PUBLICA",
    "escola_publica"
).count().orderBy(
    "IN_PUBLICA"
).show()


# no código acima foi verificado que na base realmente só há escolas públicas, com a contagem de IN_PUBLICA correspondendo a todas as escolas

+----------+--------------+-----+
|IN_PUBLICA|escola_publica|count|
+----------+--------------+-----+
|         1|       Pública|70151|
+----------+--------------+-----+



## 1.4. Coluna localizacao

criar a coluna localizacao, informando se é rural ou urbana a partir da coluna ID_LOCALIZACAO. sendo 1 Urbana e 2 Rural

In [0]:

# 5. Criar a coluna localizacao
# ID_LOCALIZACAO = 1 → Urbana
# ID_LOCALIZACAO = 2 → Rural

df_saeb = df_saeb.withColumn(
    "localizacao",
    when(col("ID_LOCALIZACAO") == 1, "Urbana")
    .when(col("ID_LOCALIZACAO") == 2, "Rural")
    .otherwise("Não identificado")
)

### 1.4.1. Qualidade dos Dados

In [0]:
df_saeb.select("localizacao").distinct().show() 

# Tratamento para verificar se há algum valor faltante. Como só há valores Urbana  e Rural, a transformação foi correta

+-----------+
|localizacao|
+-----------+
|     Urbana|
|      Rural|
+-----------+



## 1.5. Criação de colunas escola_avaliada

Criação de colunas de FLAG, informando se a escola participou ou não do exame naquela faixa de ensino

In [0]:
# ============================================================
# CRIAÇÃO DAS FLAGS DE ETAPA DE ENSINO
# ============================================================

#Para facilitar para o usuário, foram criadas as colunas para verificar em que anos escolares as escolas foram avaliadas, facilitando filtragens

# Flag do 5º ano

df_saeb = df_saeb.withColumn(
    "Escola_Avaliada_5EF",
    when(col("MEDIA_5EF_LP").isNotNull(), "Sim")
    .otherwise("Não")
)


# Flag do 9º ano


df_saeb = df_saeb.withColumn(
    "Escola_Avaliada_9EF",
    when(col("MEDIA_9EF_LP").isNotNull(), "Sim")
    .otherwise("Não")
)


# Flag do Ensino Médio


df_saeb = df_saeb.withColumn(
    "Escola_Avaliada_EM",
    when(col("MEDIA_EM_LP").isNotNull(), "Sim")
    .otherwise("Não")
)

### 1.5.1. Qualidade dos Dados

In [0]:
# ============================================================
# VALIDAÇÃO DAS FLAGS DE ETAPA
# ============================================================

print("Distribuição - 5º ano")
df_saeb.groupBy("Escola_Avaliada_5EF").count().orderBy("Escola_Avaliada_5EF").show()

print("Distribuição - 9º ano")
df_saeb.groupBy("Escola_Avaliada_9EF").count().orderBy("Escola_Avaliada_9EF").show()

print("Distribuição - Ensino Médio")
df_saeb.groupBy("Escola_Avaliada_EM").count().orderBy("Escola_Avaliada_EM").show()


# Tratamento feito corretamente, com as flags funcionando

Distribuição - 5º ano
+-------------------+-----+
|Escola_Avaliada_5EF|count|
+-------------------+-----+
|                Não|28870|
|                Sim|41281|
+-------------------+-----+

Distribuição - 9º ano
+-------------------+-----+
|Escola_Avaliada_9EF|count|
+-------------------+-----+
|                Não|39071|
|                Sim|31080|
+-------------------+-----+

Distribuição - Ensino Médio
+------------------+-----+
|Escola_Avaliada_EM|count|
+------------------+-----+
|               Não|55720|
|               Sim|14431|
+------------------+-----+



##1.6. Colunas de proficiência

Criação de colunas de proficiencia por ano escolar e disciplina

In [0]:

# ============================================================
# CLASSIFICAÇÃO DA PROFICIÊNCIA
# ============================================================
# As médias são classificadas conforme:
# - Etapa de ensino: 5º EF, 9º EF ou Ensino Médio
# - Disciplina: Língua Portuguesa (LP) ou Matemática (MT)
#
# Valores nulos permanecem como "Sem resultado".
# ============================================================


# ------------------------------------------------------------
# 5º ANO - LÍNGUA PORTUGUESA
# ------------------------------------------------------------
df_saeb = df_saeb.withColumn(
    "PROFICIENCIA_5EF_LP",
    when(col("MEDIA_5EF_LP").isNull(), "Sem resultado")
    .when(col("MEDIA_5EF_LP") <= 150, "Abaixo do Básico")
    .when(col("MEDIA_5EF_LP") <= 200, "Básico")
    .when(col("MEDIA_5EF_LP") <= 250, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# 5º ANO - MATEMÁTICA
# ------------------------------------------------------------
df_saeb = df_saeb.withColumn(
    "PROFICIENCIA_5EF_MT",
    when(col("MEDIA_5EF_MT").isNull(), "Sem resultado")
    .when(col("MEDIA_5EF_MT") <= 175, "Abaixo do Básico")
    .when(col("MEDIA_5EF_MT") <= 225, "Básico")
    .when(col("MEDIA_5EF_MT") <= 275, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# 9º ANO - LÍNGUA PORTUGUESA
# ------------------------------------------------------------
df_saeb = df_saeb.withColumn(
    "PROFICIENCIA_9EF_LP",
    when(col("MEDIA_9EF_LP").isNull(), "Sem resultado")
    .when(col("MEDIA_9EF_LP") <= 225, "Abaixo do Básico")
    .when(col("MEDIA_9EF_LP") <= 275, "Básico")
    .when(col("MEDIA_9EF_LP") <= 325, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# 9º ANO - MATEMÁTICA
# ------------------------------------------------------------
df_saeb = df_saeb.withColumn(
    "PROFICIENCIA_9EF_MT",
    when(col("MEDIA_9EF_MT").isNull(), "Sem resultado")
    .when(col("MEDIA_9EF_MT") <= 225, "Abaixo do Básico")
    .when(col("MEDIA_9EF_MT") <= 300, "Básico")
    .when(col("MEDIA_9EF_MT") <= 350, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# ENSINO MÉDIO - LÍNGUA PORTUGUESA
# ------------------------------------------------------------
df_saeb = df_saeb.withColumn(
    "PROFICIENCIA_EM_LP",
    when(col("MEDIA_EM_LP").isNull(), "Sem resultado")
    .when(col("MEDIA_EM_LP") <= 250, "Abaixo do Básico")
    .when(col("MEDIA_EM_LP") <= 300, "Básico")
    .when(col("MEDIA_EM_LP") <= 350, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# ENSINO MÉDIO - MATEMÁTICA
# ------------------------------------------------------------
df_saeb = df_saeb.withColumn(
    "PROFICIENCIA_EM_MT",
    when(col("MEDIA_EM_MT").isNull(), "Sem resultado")
    .when(col("MEDIA_EM_MT") <= 275, "Abaixo do Básico")
    .when(col("MEDIA_EM_MT") <= 350, "Básico")
    .when(col("MEDIA_EM_MT") <= 400, "Adequado")
    .otherwise("Avançado")
)

### 1.6.1. Qualidade dos Dados

In [0]:
# ============================================================
# VALIDAÇÃO DAS CLASSIFICAÇÕES DE PROFICIÊNCIA
# ============================================================


# Realizada verificação para saber quais dados foram gerados para as 6 colunas de proficiência criadas. Quantidade "Sem Resultado" em linha com o observado na etapa 05.1., mostrando consistência entre dados gerados


print("5º EF - Língua Portuguesa")
df_saeb.groupBy("PROFICIENCIA_5EF_LP").count().show()

print("5º EF - Matemática")
df_saeb.groupBy("PROFICIENCIA_5EF_MT").count().show()

print("9º EF - Língua Portuguesa")
df_saeb.groupBy("PROFICIENCIA_9EF_LP").count().show()

print("9º EF - Matemática")
df_saeb.groupBy("PROFICIENCIA_9EF_MT").count().show()

print("Ensino Médio - Língua Portuguesa")
df_saeb.groupBy("PROFICIENCIA_EM_LP").count().show()

print("Ensino Médio - Matemática")
df_saeb.groupBy("PROFICIENCIA_EM_MT").count().show()

5º EF - Língua Portuguesa
+-------------------+-----+
|PROFICIENCIA_5EF_LP|count|
+-------------------+-----+
|             Básico|15937|
|           Adequado|23207|
|      Sem resultado|28870|
|   Abaixo do Básico|  637|
|           Avançado| 1500|
+-------------------+-----+

5º EF - Matemática
+-------------------+-----+
|PROFICIENCIA_5EF_MT|count|
+-------------------+-----+
|             Básico|23025|
|   Abaixo do Básico| 2758|
|           Adequado|14552|
|      Sem resultado|28870|
|           Avançado|  946|
+-------------------+-----+

9º EF - Língua Portuguesa
+-------------------+-----+
|PROFICIENCIA_9EF_LP|count|
+-------------------+-----+
|      Sem resultado|39071|
|             Básico|23438|
|           Adequado| 4093|
|   Abaixo do Básico| 3483|
|           Avançado|   66|
+-------------------+-----+

9º EF - Matemática
+-------------------+-----+
|PROFICIENCIA_9EF_MT|count|
+-------------------+-----+
|      Sem resultado|39071|
|             Básico|26250|
|   Abaixo 

In [0]:
df_saeb.select(
    "MEDIA_5EF_LP",
    "PROFICIENCIA_5EF_LP",
    "MEDIA_5EF_MT",
    "PROFICIENCIA_5EF_MT",
    "MEDIA_9EF_LP",
    "PROFICIENCIA_9EF_LP",
    "MEDIA_9EF_MT",
    "PROFICIENCIA_9EF_MT",
    "MEDIA_EM_LP",
    "PROFICIENCIA_EM_LP",
    "MEDIA_EM_MT",
    "PROFICIENCIA_EM_MT"
).show(20)

# Visualização das colunas geradas para fazer um check visual básico nos resultados obtidos

+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+-----------+------------------+-----------+------------------+
|MEDIA_5EF_LP|PROFICIENCIA_5EF_LP|MEDIA_5EF_MT|PROFICIENCIA_5EF_MT|MEDIA_9EF_LP|PROFICIENCIA_9EF_LP|MEDIA_9EF_MT|PROFICIENCIA_9EF_MT|MEDIA_EM_LP|PROFICIENCIA_EM_LP|MEDIA_EM_MT|PROFICIENCIA_EM_MT|
+------------+-------------------+------------+-------------------+------------+-------------------+------------+-------------------+-----------+------------------+-----------+------------------+
|      166.45|             Básico|       175.9|             Básico|        NULL|      Sem resultado|        NULL|      Sem resultado|       NULL|     Sem resultado|       NULL|     Sem resultado|
|      163.69|             Básico|      162.94|   Abaixo do Básico|      235.83|             Básico|      240.96|             Básico|       NULL|     Sem resultado|       NULL|     Sem resultado|
|      222.33|      

## 1.7 e 1.8. Apagar colunas codificadas que não serão utilizadas pelos usuários



In [0]:
# ============================================================
# REMOÇÃO DAS COLUNAS DE CÓDIGO
# ============================================================
# As colunas abaixo foram utilizadas como referência para criar
# as respectivas colunas descritivas e não são mais necessárias
# para a análise final.
#
# As colunas MEDIA_* são mantidas, pois contêm as médias de
# proficiência utilizadas nas análises.
# ============================================================


df_saeb = df_saeb.drop(
    "ID_REGIAO",
    "ID_UF",
    "ID_AREA",
    "IN_PUBLICA",
    "ID_LOCALIZACAO"
)

## 1.10. Visualização Final da tabela dados_saeb

In [0]:
display(df_saeb.limit(10))

ID_SAEB,ID_MUNICIPIO,ID_ESCOLA,PC_FORMACAO_DOCENTE_INICIAL,PC_FORMACAO_DOCENTE_FINAL,PC_FORMACAO_DOCENTE_MEDIO,NIVEL_SOCIO_ECONOMICO,NU_MATRICULADOS_CENSO_5EF,NU_PRESENTES_5EF,TAXA_PARTICIPACAO_5EF,NIVEL_0_LP5,NIVEL_1_LP5,NIVEL_2_LP5,NIVEL_3_LP5,NIVEL_4_LP5,NIVEL_5_LP5,NIVEL_6_LP5,NIVEL_7_LP5,NIVEL_8_LP5,NIVEL_9_LP5,NIVEL_0_MT5,NIVEL_1_MT5,NIVEL_2_MT5,NIVEL_3_MT5,NIVEL_4_MT5,NIVEL_5_MT5,NIVEL_6_MT5,NIVEL_7_MT5,NIVEL_8_MT5,NIVEL_9_MT5,NIVEL_10_MT5,NU_MATRICULADOS_CENSO_9EF,NU_PRESENTES_9EF,TAXA_PARTICIPACAO_9EF,NIVEL_0_LP9,NIVEL_1_LP9,NIVEL_2_LP9,NIVEL_3_LP9,NIVEL_4_LP9,NIVEL_5_LP9,NIVEL_6_LP9,NIVEL_7_LP9,NIVEL_8_LP9,NIVEL_0_MT9,NIVEL_1_MT9,NIVEL_2_MT9,NIVEL_3_MT9,NIVEL_4_MT9,NIVEL_5_MT9,NIVEL_6_MT9,NIVEL_7_MT9,NIVEL_8_MT9,NIVEL_9_MT9,NU_MATRICULADOS_CENSO_EM,NU_PRESENTES_EM,TAXA_PARTICIPACAO_EM,NIVEL_0_LPEM,NIVEL_1_LPEM,NIVEL_2_LPEM,NIVEL_3_LPEM,NIVEL_4_LPEM,NIVEL_5_LPEM,NIVEL_6_LPEM,NIVEL_7_LPEM,NIVEL_8_LPEM,NIVEL_0_MTEM,NIVEL_1_MTEM,NIVEL_2_MTEM,NIVEL_3_MTEM,NIVEL_4_MTEM,NIVEL_5_MTEM,NIVEL_6_MTEM,NIVEL_7_MTEM,NIVEL_8_MTEM,NIVEL_9_MTEM,NIVEL_10_MTEM,MEDIA_5EF_LP,MEDIA_5EF_MT,MEDIA_9EF_LP,MEDIA_9EF_MT,MEDIA_EM_LP,MEDIA_EM_MT,ESTADO,area,escola_publica,localizacao,Escola_Avaliada_5EF,Escola_Avaliada_9EF,Escola_Avaliada_EM,PROFICIENCIA_5EF_LP,PROFICIENCIA_5EF_MT,PROFICIENCIA_9EF_LP,PROFICIENCIA_9EF_MT,PROFICIENCIA_EM_LP,PROFICIENCIA_EM_MT
2023,6322170,61400934,100.0,43.2,null,N�vel V,14,15,107.14,"13,33",20.0,"26,67","13,33","13,33","13,33",0.0,0.0,0.0,0.0,6.67,26.67,20.0,13.33,20.0,13.33,0.0,0.0,0.0,0.0,0.0,15,10,66.67,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,166.45,175.9,null,null,null,null,Rondônia,Interior,Pública,Urbana,Sim,Não,Não,Básico,Básico,Sem resultado,Sem resultado,Sem resultado,Sem resultado
2023,6322170,61403177,100.0,55.6,null,N�vel IV,19,19,100.0,5.26,31.58,26.32,26.32,5.26,5.26,0.0,0.0,0.0,0.0,15.79,31.58,21.05,15.79,5.26,0.0,5.26,5.26,0.0,0.0,0.0,30,25,83.33,12.0,40.0,4.0,28.0,16.0,0.0,0.0,0.0,0.0,8.0,32.0,20.0,20.0,12.0,4.0,4.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,163.69,162.94,235.83,240.96,null,null,Rondônia,Interior,Pública,Urbana,Sim,Sim,Não,Básico,Abaixo do Básico,Básico,Básico,Sem resultado,Sem resultado
2023,6322170,61412274,56.3,85.2,91.1,N�vel IV,31,31,100.0,0,12.9,9.68,12.9,9.68,19.35,22.58,6.45,6.45,0.0,0.0,3.23,3.23,6.45,19.35,25.81,22.58,9.68,6.45,3.23,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,111,111,100.0,32.41,14.87,20.06,14.35,11.99,4.52,1.8,0.0,0.0,26.09,21.91,22.31,13.68,13.39,0.9,0.9,0.82,0.0,0.0,0.0,222.33,244.1,null,null,253.38,254.41,Rondônia,Interior,Pública,Urbana,Sim,Não,Sim,Adequado,Adequado,Sem resultado,Sem resultado,Básico,Abaixo do Básico
2023,6322170,61416961,100.0,45.5,null,N�vel IV,18,13,72.22,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,12,11,91.67,18.18,18.18,0.0,18.18,27.27,18.18,0.0,0.0,0.0,9.09,9.09,36.36,18.18,27.27,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,257.79,249.24,null,null,Rondônia,Interior,Pública,Rural,Não,Sim,Não,Sem resultado,Sem resultado,Básico,Básico,Sem resultado,Sem resultado
2023,6322170,61420915,null,73.2,70.3,N�vel IV,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,111,106,95.5,12.42,17.84,21.69,16.89,16.13,9.44,5.59,0.0,0.0,11.34,14.16,18.1,22.47,11.25,11.41,9.4,1.86,0.0,0.0,66,61,92.42,11.06,6.42,14.9,29.08,16.6,12.59,9.36,0.0,0.0,8.38,11.41,17.93,27.66,19.63,10.26,3.13,1.61,0.0,0.0,0.0,null,null,249.09,259.25,288.04,285.14,Rondônia,Interior,Pública,Urbana,Não,Sim,Sim,Sem resultado,Sem resulta

# 2. Qualidade dos dados - teste de chave entre tabelas atributos_estados, dados_socioeconomicos_estados e dados_saeb

Foi validado o relacionamento entre a dimensão de estados (tabela atributos_estados) e as tabelas do projeto para garantir a integridade dos dados e evitar duplicações nos cruzamentos. Para isso, verificou-se a unicidade da chave Estado na dimensão, a correspondência dos estados nas demais tabelas e se os relacionamentos não aumentavam indevidamente a quantidade de registros.

In [0]:

df_atributos_estados = spark.table("`mvp-eng-dados-puc-rio`.bronze.atributos_estados")
df_estados =  spark.table("`mvp-eng-dados-puc-rio`.bronze.dados_socioeconomicos_estados")

In [0]:
df_atributos_estados.groupBy("Estado") \
    .count() \
    .filter(col("count") > 1) \
    .show()


    #Foi verificado que não há nenhum estado duplicado na nossa tabela dimensão, garantindo que a chave primária é única

+------+-----+
|Estado|count|
+------+-----+
+------+-----+



In [0]:
df_estados_sem_dimensao = (
    df_estados.alias("e")
    .join(
        df_atributos_estados.alias("d"),
        col("e.UFN") == col("d.Estado"),
        "left_anti"
    )
)

print(
    "Estados de df_estados sem correspondência na dimensão:",
    df_estados_sem_dimensao.select("UFN").distinct().count()
)

df_estados_sem_dimensao.select("UFN").distinct().show()


# Na junção da tabela dimensão com a tabela df_estados foi verificado que não há nenhum estado contido em uma tabela que não tenha correspondência

Estados de df_estados sem correspondência na dimensão: 0
+---+
|UFN|
+---+
+---+



In [0]:
qtd_estados_antes = df_estados.count()

df_estados_join = (
    df_estados.alias("e")
    .join(
        df_atributos_estados.alias("d"),
        col("e.UFN") == col("d.Estado"),
        "left"
    )
)

qtd_estados_depois = df_estados_join.count()

print("df_estados antes:", qtd_estados_antes)
print("df_estados depois:", qtd_estados_depois)
print("Diferença:", qtd_estados_depois - qtd_estados_antes)


    #Foi verificado que não não houve incremento de linhas ao realizar um join entre as tabelas, mostrando que a relação 1 > N está funcionando

df_estados antes: 81
df_estados depois: 81
Diferença: 0


In [0]:
df_saeb_sem_dimensao = (
    df_saeb.alias("s")
    .join(
        df_atributos_estados.alias("d"),
        col("s.ESTADO") == col("d.Estado"),
        "left_anti"
    )
)

print(
    "Estados do SAEB sem correspondência na dimensão:",
    df_saeb_sem_dimensao.select("ESTADO").distinct().count()
)

df_saeb_sem_dimensao.select("ESTADO").distinct().show()


# Na junção da tabela dimensão com a tabela df_saeb foi verificado que não há nenhum estado contido em uma tabela que não tenha correspondência

Estados do SAEB sem correspondência na dimensão: 0
+------+
|ESTADO|
+------+
+------+



In [0]:
qtd_saeb_antes = df_saeb.count()

df_saeb_join = (
    df_saeb.alias("s")
    .join(
        df_atributos_estados.alias("d"),
        col("s.ESTADO") == col("d.Estado"),
        "left"
    )
)

qtd_saeb_depois = df_saeb_join.count()

print("df_saeb antes:", qtd_saeb_antes)
print("df_saeb depois:", qtd_saeb_depois)
print("Diferença:", qtd_saeb_depois - qtd_saeb_antes)

    #Foi verificado que não não houve incremento de linhas ao realizar um join entre a tabela dimensão e a tabela df_saeb , mostrando que a relação 1 > N está funcionando

df_saeb antes: 70151
df_saeb depois: 70151
Diferença: 0


# 3. Upload na camada silver

Como as tabelas de atributos e dados_socioeconomicos não precisaram de ajustes, somente a tabela de resultados_saeb sera inserida na camada Silver, com as demais sendo mantidas na bronze, a fim de evitar duplicação de dados

In [0]:
spark.sql("USE CATALOG `mvp-eng-dados-puc-rio`")
spark.sql("USE SCHEMA `silver`")


In [0]:
df_saeb.write.format("delta").mode("overwrite").saveAsTable("resultados_saeb")